# My run crashed. Where do I find out why?

This page shows how to tell whether a run really finished, and how to find the error message
when it did not. Replace `$CASE` with your own case name.

## Three quick ways to tell if your run crashed

"No longer listed in `qstat`" is not proof of success: a crashed job disappears from the queue
too. Check one of these instead:

1. **Archive directory**: did new history files show up under
   `/glade/derecho/scratch/$USER/archive/$CASE/`? If it is still empty (or missing the month
   you expected), the run probably did not finish.
2. **Restart files**: are there `*.r.*` and `rpointer.*` files in the run directory (or under
   `rest/` once archived) with a date matching the expected end of your run? If they are
   missing, the model never reached a clean stopping point.
3. **`CaseStatus`**: does the file `CaseStatus` in the case directory end with
   `case.run success` and `st_archive success`? See
   [The 4 commands to run CESM](../04.CESM-Workflow-Quickstart/4.1_four_commands.ipynb) for a
   worked example. A failed run shows `error` instead:

   ```text
   2026-09-02 14:32:09: case.run starting
   ---------------------------------------------------
   2026-09-02 14:33:41: case.run error
   ```

   The timestamp of the `error` line tells you roughly when the run died, and which step
   (`case.setup`, `case.build`, or `case.run`) to focus on.

## Find the error in the log files

If your run crashed, the **log files** in the run directory (`$RUNDIR`) are the place to look.
A crashed run usually does not move its logs to `logs/` in the case directory, so go to the run
directory directly:

```bash
cd /glade/derecho/scratch/$USER/$CASE/run
ls -ltr *log*
```

`ls -ltr` lists the log files oldest to newest, so the component that crashed (the one whose log
stopped updating) is usually near the bottom.

Then search for `ERROR` or `ABORT` instead of just reading the last few lines:

```bash
grep -in "error\|abort" cesm.log.* | tail -20
```

:::{warning}
The very bottom of a log file is often just the batch system's normal shutdown message, not the
actual cause. The real error is usually a little *earlier* in the file, around the last time
step the model completed.
:::

## General troubleshooting checklist

1. Run `qstat -u $USER`. Did the job start at all, or is it still waiting in the queue?
2. Run through the three checks above to confirm the run really failed, and which step failed.
3. If the **build** failed, look at the build logs in the build directory:
   `/glade/derecho/scratch/$USER/$CASE/bld/*.bldlog.*`.
4. If the **run** failed, work through the log files as described above, starting with
   `cesm.log.*`.
5. Re-read [The xml files and run length](../05.Run-length/5.1_xml_files.ipynb). Many crashes
   come from a `STOP_N`/`RESUBMIT` combination that needs more wall-clock time than the job was
   given.
6. Still stuck? See [Where to find help](9.3_getting_help.ipynb).